In [4]:
import zipfile
import io
import yaml
import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory
from quantum_lake_student.formats import iter_b8_records, parse_01_records

settings = Settings.from_environment()
client = minio_client(settings)

In [5]:
def read_zip_member(bronze_key, member_name):
    response = client.get_object(settings.s3_bucket, bronze_key)
    data = response.read()
    response.close()
    response.release_conn()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        return zf.read(member_name)

def list_zip_members(bronze_key):
    response = client.get_object(settings.s3_bucket, bronze_key)
    data = response.read()
    response.close()
    response.release_conn()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        return [(name, zf.getinfo(name).file_size) for name in zf.namelist()]

In [6]:
google_key = "bronze/source=google_qec/google-surface-code-curated.zip"
syndromes_key = "bronze/source=qec_syndromes/syndromes_dataset.zip"
qasmbench_key = "bronze/source=qasmbench/qasmbench-qec.zip"

In [7]:
exp_dir = "surface_code_bX_d3_r25_center_3_5"

props = yaml.safe_load(read_zip_member(google_key, f"{exp_dir}/properties.yml"))
print("Properties:", props)

meas_bytes = read_zip_member(google_key, f"{exp_dir}/measurements.b8")
first_measurement = next(iter_b8_records(meas_bytes, bits_per_record=props["circuit_measurements"]))
print("\nFirst measurement record (len={}):".format(len(first_measurement)), first_measurement)

det_bytes = read_zip_member(google_key, f"{exp_dir}/detection_events.b8")
first_detection = next(iter_b8_records(det_bytes, bits_per_record=props["circuit_detectors"]))
print("\nFirst detection record (len={}):".format(len(first_detection)), first_detection)

obs_bytes = read_zip_member(google_key, f"{exp_dir}/obs_flips_actual.01")
obs_flips = parse_01_records(obs_bytes)
print("\nFirst 10 obs_flips_actual values:", obs_flips[:10])
print("Total shots:", len(obs_flips))

Properties: {'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}

First measurement record (len=209): (1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1

In [8]:
syn_bytes = read_zip_member(syndromes_key, "d-3_pfr-0.000010_nb-10M.csv")
syn_df = pd.read_csv(io.BytesIO(syn_bytes))
print(syn_df.shape)
syn_df.head(3)

(68, 3)


,labels,syndromes,quantity
0,0,"((0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",9987291
1,0,"((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0,...",486
2,1,"((0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,...",476


In [9]:
members = list_zip_members(qasmbench_key)
for name, size in members:
    print(f"{name}  ({size:,} bytes)")

LICENSE  (2,066 bytes)
NOTICE  (1,285 bytes)
README.md  (27,140 bytes)
qelib1.inc  (4,090 bytes)
small/error_correctiond3_n5/README.md  (571 bytes)
small/error_correctiond3_n5/error_correctiond3_n5.png  (62,789 bytes)
small/error_correctiond3_n5/error_correctiond3_n5.qasm  (1,517 bytes)
small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm  (3,304 bytes)
small/error_correctiond3_n5/res_error_correctiond3_n5.png  (16,042 bytes)
small/qec_en_n5/README.md  (514 bytes)
small/qec_en_n5/qec_en_n5.png  (29,104 bytes)
small/qec_en_n5/qec_en_n5.qasm  (530 bytes)
small/qec_en_n5/qec_en_n5_transpiled.qasm  (778 bytes)
small/qec_en_n5/res_qec_en_n5.png  (12,183 bytes)
small/qec_sm_n5/README.md  (480 bytes)
small/qec_sm_n5/qec_sm_n5.png  (25,706 bytes)
small/qec_sm_n5/qec_sm_n5.qasm  (377 bytes)
small/qec_sm_n5/qec_sm_n5_transpiled.qasm  (341 bytes)
small/qec_sm_n5/res_qec_sm_n5.png  (12,085 bytes)


In [10]:
print(read_zip_member(qasmbench_key, "small/error_correctiond3_n5/error_correctiond3_n5.qasm").decode())

// Error correction: distance-three 5-qubit code, from the paper "Benchmarking gate-based quantum computers" by K. Michielsen et al.

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0];
h q[1];
id q[2];
h q[3];
h q[4];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[4],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
sdg q[4];
cx q[4],q[2];
h q[2];
cx q[4],q[2];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
cx q[3],q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
h q[0];
h q[2];
cx q[0],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[3],q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
h q[1];
h q[2];
cx q[1],q[2];
cx q[3],q[2];
cx q[0],q[2];
h q[3];
h q[4];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
cx q[0],q[2];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
h q[2];
h q[3];
cx q[3],q[2];
cx q[4],q[2];
h q[2

In [11]:
print("=== qec_en_n5.qasm ===")
print(read_zip_member(qasmbench_key, "small/qec_en_n5/qec_en_n5.qasm").decode())
print("\n\n=== qec_sm_n5.qasm ===")
print(read_zip_member(qasmbench_key, "small/qec_sm_n5/qec_sm_n5.qasm").decode())

=== qec_en_n5.qasm ===
// Name of Experiment: Encoder into bit-flip code with parity checks (qubits 0,1,3) v2

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[2];
t q[2];
h q[2];
h q[0];
h q[1];
h q[2];
cx q[1], q[2];
cx q[0], q[2];
h q[0];
h q[1];
h q[3];
cx q[3], q[2];
h q[2];
h q[3];
cx q[3], q[2];
cx q[0], q[2];
cx q[1], q[2];
h q[2];
h q[4];
cx q[4], q[2];
h q[2];
h q[4];
cx q[4], q[2];
cx q[1], q[2];
cx q[3], q[2];


measure q[2] -> c[2];
measure q[4] -> c[4];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[3] -> c[3];



=== qec_sm_n5.qasm ===
// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



In [12]:
print("=== qec_en_n5.qasm ===")
print(read_zip_member(qasmbench_key, "small/qec_en_n5/qec_en_n5.qasm").decode())
print("\n\n=== qec_sm_n5.qasm ===")
print(read_zip_member(qasmbench_key, "small/qec_sm_n5/qec_sm_n5.qasm").decode())

=== qec_en_n5.qasm ===
// Name of Experiment: Encoder into bit-flip code with parity checks (qubits 0,1,3) v2

OPENQASM 2.0;
include "qelib1.inc";

qreg q[5];
creg c[5];

h q[2];
t q[2];
h q[2];
h q[0];
h q[1];
h q[2];
cx q[1], q[2];
cx q[0], q[2];
h q[0];
h q[1];
h q[3];
cx q[3], q[2];
h q[2];
h q[3];
cx q[3], q[2];
cx q[0], q[2];
cx q[1], q[2];
h q[2];
h q[4];
cx q[4], q[2];
h q[2];
h q[4];
cx q[4], q[2];
cx q[1], q[2];
cx q[3], q[2];


measure q[2] -> c[2];
measure q[4] -> c[4];
measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[3] -> c[3];



=== qec_sm_n5.qasm ===
// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



In [14]:
for family in ["error_correctiond3_n5", "qec_en_n5", "qec_sm_n5"]:
    print(f"\n=== {family}/README.md ===")
    print(read_zip_member(qasmbench_key, f"small/{family}/README.md").decode())


=== error_correctiond3_n5/README.md ===
# Application: error_correctiond3_n5
- Qubit Count : 5
- Circuit Depth : 78
- Circuit Width : 5
- Retention Lifespan : 4.356708826689592
- Gate Density : 0.41794871794871796
- Dual Gate Count : 49
- Measurement Density : 1.1932293478247384
- Size Factor : 5.093750200806762
- Gate Count : 114
- Entanglement Variance : 1.390583966187302
- Communication Supermarq : 0.5
- Measurement Supermarq : 0.0
- Depth Supermarq : 0.9591836734693877
- Entanglement Supermarq : 0.4298245614035088
- Parallelism Supermarq : 0.3157894736842105
- Liveness Supermarq : 0.4307692307692308


=== qec_en_n5/README.md ===
# Application: qec_en_n5
- Qubit Count : 5
- Circuit Depth : 18
- Circuit Width : 5
- Retention Lifespan : 2.8903717578961645
- Gate Density : 0.3888888888888889
- Dual Gate Count : 10
- Measurement Density : 0.899961934066053
- Size Factor : 4.174387269895637
- Gate Count : 25
- Entanglement Variance : 1.0396994062531653
- Communication Supermarq : 0.4
- 